# Tests

This notebook consolidates the PENRS test suite from eight source files: `test_arbiter`, `test_cache`, `test_pipeline`, `test_rate_limit`, `test_router`, `test_step1`, `test_step2`, and `test_worker`. Tests are organized into labeled sections by source module, and the final cell invokes `pytest` to run the full suite and print a pass/fail summary.

In [ ]:
# Standard library
import asyncio
import importlib
import json
import os
import uuid
from datetime import datetime, timedelta, timezone
from enum import Enum
from pathlib import Path
from unittest.mock import patch

# Third-party
import httpx
import pytest

# Project — utility module (consolidates penrs_cache, penrs_http, penrs_rate_limit, penrs_router)
import utils

# Aliases so existing test helpers retain their patching targets:
#   _reload_cache_module reloads utils and accesses utils.cache_*, utils.logger, etc.
#   Step 2 tests patch utils.httpx.AsyncClient, utils.asyncio.sleep, utils.logger, etc.
penrs_cache = utils
penrs_http = utils

# Notebook-resident symbols (migrated from deleted .py source files):
#   PENRSWorker, truncate_for_context  ← worker_nodes.ipynb
#   ArbiterAgent, ARBITER_SYSTEM_PROMPT, run_all_workers, run_penrs  ← orchestrator.ipynb
%run worker_nodes.ipynb
%run orchestrator.ipynb

# Project — symbols from utils
from utils import DOCUMENT_API_ROUTING, DocumentType, penrs_fetch_document

## Arbiter Tests

In [2]:
def _worker_result(
    *,
    name: str,
    weight: float,
    signal_density: float,
    score: float,
    star_rating: int | None = None,
    summary: str = "",
) -> dict:
    result_payload = {"score": score, "summary": summary}
    if star_rating is not None:
        result_payload["star_rating"] = star_rating
    return {
        "status": "available",
        "worker": {
            "name": name,
            "weight": weight,
            "signal_density": signal_density,
        },
        "result": result_payload,
    }


def test_system_prompt_contains_required_role_and_mandatory_contradictions():
    assert "Lead Portfolio Manager" in ARBITER_SYSTEM_PROMPT
    assert "Lipstick on a Pig" in ARBITER_SYSTEM_PROMPT
    assert "Bailing Out" in ARBITER_SYSTEM_PROMPT
    assert "Dilute and Delay" in ARBITER_SYSTEM_PROMPT


def test_arbiter_validates_required_worker_schema_fields():
    arbiter = ArbiterAgent()
    bad_worker = {
        "status": "available",
        "worker": {
            "name": "MissingWeight",
            "signal_density": 0.8,
        },
        "result": {"score": 0.3},
    }

    with pytest.raises(ValueError, match="worker.weight"):
        arbiter.evaluate([bad_worker])


def test_arbiter_clamps_scores_and_applies_star_rating_weights():
    arbiter = ArbiterAgent()
    workers = [
        _worker_result(
            name="WorkerA",
            weight=2.0,
            signal_density=0.9,
            score=1.8,
            star_rating=5,
        ),
        _worker_result(
            name="WorkerB",
            weight=1.5,
            signal_density=0.2,
            score=-2.4,
            star_rating=2,
        ),
    ]

    report = arbiter.evaluate(workers)
    worker_scores = {entry["name"]: entry for entry in report["worker_scores"]}

    assert worker_scores["WorkerA"]["normalized_score"] == 1.0
    assert worker_scores["WorkerA"]["effective_weight"] == 2.0
    assert worker_scores["WorkerA"]["weighted_score"] == 2.0

    assert worker_scores["WorkerB"]["normalized_score"] == -1.0
    assert worker_scores["WorkerB"]["effective_weight"] == 0.6
    assert worker_scores["WorkerB"]["weighted_score"] == -0.6

    assert report["weighted_score"] == pytest.approx(1.4 / 2.6, rel=1e-6)


def test_arbiter_returns_json_schema_with_contradiction_flags_and_severities():
    arbiter = ArbiterAgent()
    workers = [
        _worker_result(
            name="NarrativeWorker",
            weight=1.0,
            signal_density=0.6,
            score=0.2,
            summary="Management is putting lipstick on a pig while bailing out early holders.",
        )
    ]

    report = arbiter.evaluate(workers)
    contradictions = {entry["name"]: entry for entry in report["contradictions"]}

    assert set(contradictions) == {"Lipstick on a Pig", "Bailing Out", "Dilute and Delay"}
    assert contradictions["Lipstick on a Pig"]["flagged"] is True
    assert contradictions["Lipstick on a Pig"]["severity"] == "High"
    assert contradictions["Bailing Out"]["flagged"] is True
    assert contradictions["Bailing Out"]["severity"] == "High"
    assert contradictions["Dilute and Delay"]["severity"] == "Medium"

    json.dumps(report)

## Cache Tests

In [3]:
TEST_FILES_DIR = (Path.cwd() / "Test_files").resolve()


def local_tmp_dir() -> Path:
    TEST_FILES_DIR.mkdir(parents=True, exist_ok=True)
    tmp_dir = TEST_FILES_DIR / f"test_tmp_{uuid.uuid4().hex}"
    tmp_dir.mkdir(parents=True, exist_ok=True)
    return tmp_dir.resolve()


def _reload_cache_module(monkeypatch, cache_dir: Path):
    monkeypatch.setenv("PENRS_CACHE_DIR", str(cache_dir))
    importlib.reload(penrs_cache)
    return penrs_cache


def test_cache_key_is_deterministic(monkeypatch):
    mod = _reload_cache_module(monkeypatch, local_tmp_dir() / "cache")

    key1 = mod.cache_key("alpha", "MRNA", "earnings_call", "2025-01-10")
    key2 = mod.cache_key("alpha", "MRNA", "earnings_call", "2025-01-10")
    key3 = mod.cache_key("alpha", "MRNA", "earnings_call", "2025-01-11")

    assert key1 == key2
    assert key1 != key3
    assert len(key1) == 64


def test_cache_set_writes_json_with_metadata(monkeypatch):
    mod = _reload_cache_module(monkeypatch, local_tmp_dir() / "cache")
    payload = {"headline": "Sample", "score": 0.4}

    path = mod.cache_set(
        api="alpha",
        ticker="MRNA",
        doc_type="earnings_call",
        date="2025-01-10",
        payload=payload,
    )

    assert path.is_file()
    written = json.loads(path.read_text(encoding="utf-8"))

    assert "_cached_at" in written
    assert written["_api"] == "alpha"
    assert written["_ticker"] == "MRNA"
    assert written["_doc_type"] == "earnings_call"
    assert written["payload"] == payload


def test_cache_get_returns_payload_when_fresh(monkeypatch):
    mod = _reload_cache_module(monkeypatch, local_tmp_dir() / "cache")
    payload = {"k": "v", "n": 2}
    mod.cache_set(
        api="sec",
        ticker="BIIB",
        doc_type="10q",
        date="2025-02-01",
        payload=payload,
    )

    cached = mod.cache_get(
        api="sec",
        ticker="BIIB",
        doc_type="10q",
        date="2025-02-01",
        max_age_hours=12,
    )

    assert cached == payload


def test_cache_get_returns_none_for_missing_file(monkeypatch):
    mod = _reload_cache_module(monkeypatch, local_tmp_dir() / "cache")

    with patch.object(mod.logger, "info") as log_info:
        cached = mod.cache_get(
            api="ctgov",
            ticker="SAVA",
            doc_type="clinical_trial",
            date="2025-02-01",
            max_age_hours=24,
        )

    assert cached is None
    assert log_info.call_count >= 1


def test_cache_get_returns_none_when_expired(monkeypatch):
    mod = _reload_cache_module(monkeypatch, local_tmp_dir() / "cache")
    payload = {"x": 1}
    path = mod.cache_set(
        api="openfda",
        ticker="MRNA",
        doc_type="adverse_events",
        date="2025-02-01",
        payload=payload,
    )

    expired = json.loads(path.read_text(encoding="utf-8"))
    expired["_cached_at"] = (
        datetime.now(timezone.utc) - timedelta(hours=5)
    ).isoformat()
    path.write_text(json.dumps(expired, ensure_ascii=True), encoding="utf-8")

    with patch.object(mod.logger, "info") as log_info:
        cached = mod.cache_get(
            api="openfda",
            ticker="MRNA",
            doc_type="adverse_events",
            date="2025-02-01",
            max_age_hours=1,
        )

    assert cached is None
    assert log_info.call_count >= 1


def test_cache_operations_are_logged(monkeypatch):
    mod = _reload_cache_module(monkeypatch, local_tmp_dir() / f"cache-{uuid.uuid4().hex}")

    with patch.object(mod.logger, "info") as log_info:
        mod.cache_set(
            api="pubmed",
            ticker="SRPT",
            doc_type="publication",
            date="2025-03-02",
            payload={"paper_count": 3},
        )
        result = mod.cache_get(
            api="pubmed",
            ticker="SRPT",
            doc_type="publication",
            date="2025-03-02",
            max_age_hours=1,
        )

    assert result == {"paper_count": 3}
    assert log_info.call_count >= 3

## Pipeline Tests

In [ ]:
TEST_FILES_DIR = (Path.cwd() / "Test_files").resolve()


def local_tmp_dir() -> Path:
    TEST_FILES_DIR.mkdir(parents=True, exist_ok=True)
    tmp_dir = TEST_FILES_DIR / f"test_tmp_{uuid.uuid4().hex}"
    tmp_dir.mkdir(parents=True, exist_ok=True)
    return tmp_dir.resolve()


class StubWorker:
    def __init__(
        self,
        *,
        name: str,
        weight: float,
        signal_density: float,
        score: float,
        fail: bool = False,
    ) -> None:
        self.name = name
        self.weight = weight
        self.signal_density = signal_density
        self.score = score
        self.fail = fail

    async def run(self, ticker: str, date_from: str, date_to: str) -> dict:
        if self.fail:
            raise RuntimeError(f"{self.name} exploded")
        return {
            "status": "available",
            "ticker": ticker,
            "date_from": date_from,
            "date_to": date_to,
            "worker": {
                "name": self.name,
                "weight": self.weight,
                "signal_density": self.signal_density,
            },
            "result": {
                "score": self.score,
                "summary": f"{self.name} summary",
            },
        }


class StubArbiter:
    def __init__(self) -> None:
        self.received: list[dict] | None = None

    def evaluate(self, worker_results: list[dict]) -> dict:
        self.received = list(worker_results)
        return {
            "status": "available",
            "worker_scores": [],
            "weighted_score": 0.33,
            "contradictions": [],
        }


class StubMaster:
    def __init__(self) -> None:
        self.received: dict | None = None

    async def synthesize(
        self,
        *,
        ticker: str,
        date_from: str,
        date_to: str,
        worker_results: list[dict],
        arbiter_result: dict,
    ) -> dict:
        self.received = {
            "ticker": ticker,
            "date_from": date_from,
            "date_to": date_to,
            "worker_results": worker_results,
            "arbiter_result": arbiter_result,
        }
        return {
            "status": "available",
            "final_score": arbiter_result.get("weighted_score", 0.0),
        }


def test_run_penrs_executes_pipeline_and_saves_report():
    working_dir = local_tmp_dir()
    original_cwd = Path.cwd()
    os.chdir(working_dir)
    workers = [
        StubWorker(name="W1", weight=1.0, signal_density=0.7, score=0.4),
        StubWorker(name="W2", weight=1.3, signal_density=0.8, score=0.2),
    ]
    arbiter = StubArbiter()
    master = StubMaster()

    try:
        report = asyncio.run(
            run_penrs(
                "MRNA",
                "2026-01-01",
                "2026-02-01",
                workers=workers,
                arbiter=arbiter,
                master=master,
            )
        )
    finally:
        os.chdir(original_cwd)

    assert arbiter.received is not None
    assert len(arbiter.received) == 2
    assert master.received is not None
    assert master.received["arbiter_result"]["weighted_score"] == 0.33

    report_path = Path(report["report_path"])
    assert report_path.exists()
    assert report_path.parent.name == "penrs_reports"

    saved_payload = json.loads(report_path.read_text(encoding="utf-8"))
    assert saved_payload["ticker"] == "MRNA"
    assert saved_payload["master"]["final_score"] == 0.33
    assert saved_payload["arbiter"]["status"] == "available"


def test_run_penrs_worker_failure_is_isolated():
    output_dir = local_tmp_dir()
    workers = [
        StubWorker(name="FailWorker", weight=1.0, signal_density=0.6, score=0.0, fail=True),
        StubWorker(name="GoodWorker", weight=1.0, signal_density=0.6, score=0.55),
    ]
    arbiter = StubArbiter()

    report = asyncio.run(
        run_penrs(
            "BIIB",
            "2026-01-01",
            "2026-02-01",
            workers=workers,
            arbiter=arbiter,
            report_dir=output_dir,
        )
    )

    statuses = {entry["worker"]["name"]: entry["status"] for entry in report["worker_results"]}
    assert statuses["FailWorker"] == "error"
    assert statuses["GoodWorker"] == "available"
    assert arbiter.received is not None
    assert len(arbiter.received) == 1
    assert arbiter.received[0]["worker"]["name"] == "GoodWorker"
    assert Path(report["report_path"]).exists()


def test_run_all_workers_uses_asyncio_gather_return_exceptions(monkeypatch):
    captured: dict[str, bool] = {}

    async def fake_gather(*coroutines, return_exceptions=False):
        captured["return_exceptions"] = return_exceptions
        results = []
        for coroutine in coroutines:
            try:
                results.append(await coroutine)
            except Exception as exc:
                if return_exceptions:
                    results.append(exc)
                else:
                    raise
        return results

    monkeypatch.setattr(asyncio, "gather", fake_gather)
    workers = [
        StubWorker(name="Broken", weight=1.0, signal_density=0.5, score=0.0, fail=True),
        StubWorker(name="Healthy", weight=1.0, signal_density=0.5, score=0.1),
    ]

    results = asyncio.run(run_all_workers(workers, "SRPT", "2026-01-01", "2026-02-01"))

    assert captured["return_exceptions"] is True
    statuses = {entry["worker"]["name"]: entry["status"] for entry in results}
    assert statuses["Broken"] == "error"
    assert statuses["Healthy"] == "available"

## Rate Limit Tests

In [ ]:
def _reload_rate_limit_module():
    importlib.reload(utils)
    utils._reset_rate_limit_state()
    return utils


def test_alpha_vantage_blocks_after_25_daily_calls(monkeypatch):
    mod = _reload_rate_limit_module()
    now = datetime(2026, 3, 2, 12, 0, tzinfo=timezone.utc)
    sleeps = []

    monkeypatch.setattr(mod, "_now_utc", lambda: now)
    monkeypatch.setattr(mod.time, "sleep", lambda seconds: sleeps.append(seconds))

    for _ in range(25):
        assert mod._check_rate_limit("alpha_vantage") is True
    assert mod._check_rate_limit("alpha_vantage") is False

    assert all(seconds == 12 for seconds in sleeps)


def test_alpha_vantage_sleeps_12_seconds_after_5_calls_same_minute(monkeypatch):
    mod = _reload_rate_limit_module()
    now = datetime(2026, 3, 2, 9, 30, tzinfo=timezone.utc)
    sleeps = []

    monkeypatch.setattr(mod, "_now_utc", lambda: now)
    monkeypatch.setattr(mod.time, "sleep", lambda seconds: sleeps.append(seconds))

    for _ in range(5):
        assert mod._check_rate_limit("alpha_vantage") is True
    assert mod._check_rate_limit("alpha_vantage") is True

    assert sleeps == [12]


def test_daily_counter_resets_at_midnight_boundary(monkeypatch):
    mod = _reload_rate_limit_module()
    previous_minute = datetime(2026, 3, 1, 23, 59, tzinfo=timezone.utc)
    midnight = datetime(2026, 3, 2, 0, 0, tzinfo=timezone.utc)

    mod._RATE_LIMIT_STATE["alpha_vantage"] = {
        "day_key": previous_minute.date().isoformat(),
        "daily_count": 25,
        "minute_key": previous_minute.replace(second=0, microsecond=0).isoformat(),
        "minute_count": 5,
    }

    monkeypatch.setattr(mod, "_now_utc", lambda: midnight)
    monkeypatch.setattr(mod.time, "sleep", lambda _seconds: None)

    assert mod._check_rate_limit("alpha_vantage") is True
    assert mod._RATE_LIMIT_STATE["alpha_vantage"]["daily_count"] == 1


def test_minute_counter_resets_on_new_minute_for_sec_edgar(monkeypatch):
    mod = _reload_rate_limit_module()
    now = datetime(2026, 3, 2, 11, 15, tzinfo=timezone.utc)
    next_minute = now + timedelta(minutes=1)

    monkeypatch.setattr(mod, "_now_utc", lambda: now)

    for _ in range(10):
        assert mod._check_rate_limit("sec_edgar") is True
    assert mod._check_rate_limit("sec_edgar") is False

    monkeypatch.setattr(mod, "_now_utc", lambda: next_minute)
    assert mod._check_rate_limit("sec_edgar") is True


def test_other_apis_use_configurable_rpm_limit(monkeypatch):
    mod = _reload_rate_limit_module()
    now = datetime(2026, 3, 2, 8, 45, tzinfo=timezone.utc)
    monkeypatch.setattr(mod, "_now_utc", lambda: now)

    assert mod._check_rate_limit("pubmed", rpm_limit=2) is True
    assert mod._check_rate_limit("pubmed", rpm_limit=2) is True
    assert mod._check_rate_limit("pubmed", rpm_limit=2) is False


def test_warning_logged_when_limits_approached_or_hit(monkeypatch):
    mod = _reload_rate_limit_module()
    now = datetime(2026, 3, 2, 8, 50, tzinfo=timezone.utc)
    monkeypatch.setattr(mod, "_now_utc", lambda: now)

    with patch.object(mod.logger, "warning") as log_warning:
        assert mod._check_rate_limit("pubmed", rpm_limit=2) is True
        assert mod._check_rate_limit("pubmed", rpm_limit=2) is True
        assert mod._check_rate_limit("pubmed", rpm_limit=2) is False

    assert log_warning.call_count >= 2

## Router Tests

In [6]:
def test_document_type_is_strict_enum_with_nine_values():
    assert issubclass(DocumentType, Enum)
    assert len(DocumentType) == 9
    assert set(DOCUMENT_API_ROUTING.keys()) == set(DocumentType)
    assert all(isinstance(api, str) and api for apis in DOCUMENT_API_ROUTING.values() for api in apis)


def test_penrs_fetch_document_requires_document_type_enum():
    with pytest.raises(TypeError):
        asyncio.run(
            penrs_fetch_document(
                ticker="MRNA",
                document_type="sec_10q",  # type: ignore[arg-type]
                date_range=("2026-01-01", "2026-02-01"),
                fetchers={},
            )
        )


def test_penrs_fetch_document_aggregates_multi_source_results():
    async def openfda_fetcher(ticker, date_range, document_type):
        assert ticker == "MRNA"
        assert date_range == ("2026-01-01", "2026-02-01")
        assert document_type is DocumentType.BIOMEDICAL_EVIDENCE
        return {"status": "available", "data": {"events": 3}}

    async def pubmed_fetcher(_ticker, _date_range, _document_type):
        return {"status": "available", "data": {"papers": 7}}

    result = asyncio.run(
        penrs_fetch_document(
            ticker="MRNA",
            document_type=DocumentType.BIOMEDICAL_EVIDENCE,
            date_range=("2026-01-01", "2026-02-01"),
            fetchers={
                "openfda": openfda_fetcher,
                "pubmed": pubmed_fetcher,
            },
        )
    )

    assert result["status"] == "available"
    assert result["data"]["apis_attempted"] == ["openfda", "pubmed"]
    assert result["data"]["sources"] == [
        {"api": "openfda", "data": {"events": 3}},
        {"api": "pubmed", "data": {"papers": 7}},
    ]


def test_penrs_fetch_document_returns_not_released_with_attempted_apis():
    async def sec_fetcher(_ticker, _date_range, _document_type):
        return {"status": "not_released"}

    result = asyncio.run(
        penrs_fetch_document(
            ticker="BIIB",
            document_type=DocumentType.SEC_10Q,
            date_range=("2026-01-01", "2026-02-01"),
            fetchers={"sec_edgar": sec_fetcher},
        )
    )

    assert result["status"] == "not_released"
    assert result["data"]["apis_attempted"] == ["sec_edgar"]


def test_penrs_fetch_document_handles_partial_failures():
    async def openfda_fetcher(_ticker, _date_range, _document_type):
        return {"status": "available", "data": {"events": 1}}

    async def pubmed_fetcher(_ticker, _date_range, _document_type):
        raise RuntimeError("pubmed outage")

    result = asyncio.run(
        penrs_fetch_document(
            ticker="SAVA",
            document_type=DocumentType.BIOMEDICAL_EVIDENCE,
            date_range=("2026-01-01", "2026-02-01"),
            fetchers={
                "openfda": openfda_fetcher,
                "pubmed": pubmed_fetcher,
            },
        )
    )

    assert result["status"] == "available"
    assert result["data"]["sources"] == [{"api": "openfda", "data": {"events": 1}}]
    assert result["data"]["partial_failures"] == [{"api": "pubmed", "error": "pubmed outage"}]

## Step 1 Tests

In [7]:
TEST_FILES_DIR = (Path.cwd() / "Test_files").resolve()


@pytest.fixture
def local_tmp_dir() -> Path:
    TEST_FILES_DIR.mkdir(parents=True, exist_ok=True)
    tmp_dir = TEST_FILES_DIR / f"test_tmp_{uuid.uuid4().hex}"
    tmp_dir.mkdir(parents=True, exist_ok=True)
    return tmp_dir.resolve()


def test_env_defaults(local_tmp_dir, monkeypatch):
    monkeypatch.delenv("ALPHA_VANTAGE_API_KEY", raising=False)
    monkeypatch.delenv("PENRS_CACHE_DIR", raising=False)
    monkeypatch.delenv("PENRS_LOG_DIR", raising=False)
    monkeypatch.chdir(local_tmp_dir)

    # Patch at the source so that `from dotenv import load_dotenv`
    # during reload binds the mock, not the real function.
    with patch("dotenv.load_dotenv"):
        import penrs_mcp_server
        importlib.reload(penrs_mcp_server)

        assert penrs_mcp_server.ALPHA_VANTAGE_API_KEY == "demo"
        assert penrs_mcp_server.PENRS_CACHE_DIR == (local_tmp_dir / ".penrs_cache").resolve()
        assert penrs_mcp_server.PENRS_LOG_DIR == (local_tmp_dir / ".penrs_logs").resolve()


def test_dirs_created(local_tmp_dir, monkeypatch):
    monkeypatch.setenv("PENRS_CACHE_DIR", str(local_tmp_dir / "cache"))
    monkeypatch.setenv("PENRS_LOG_DIR", str(local_tmp_dir / "logs"))
    monkeypatch.chdir(local_tmp_dir)

    import penrs_mcp_server
    importlib.reload(penrs_mcp_server)

    assert (local_tmp_dir / "cache").is_dir()
    assert (local_tmp_dir / "logs").is_dir()


def test_api_key_from_env(monkeypatch, local_tmp_dir):
    monkeypatch.setenv("ALPHA_VANTAGE_API_KEY", "TESTKEY123")
    monkeypatch.setenv("PENRS_CACHE_DIR", str(local_tmp_dir / "cache"))
    monkeypatch.setenv("PENRS_LOG_DIR", str(local_tmp_dir / "logs"))
    monkeypatch.chdir(local_tmp_dir)

    import penrs_mcp_server
    importlib.reload(penrs_mcp_server)

    assert penrs_mcp_server.ALPHA_VANTAGE_API_KEY == "TESTKEY123"


def test_mcp_server_named(local_tmp_dir, monkeypatch):
    monkeypatch.setenv("PENRS_CACHE_DIR", str(local_tmp_dir / "cache"))
    monkeypatch.setenv("PENRS_LOG_DIR", str(local_tmp_dir / "logs"))
    monkeypatch.chdir(local_tmp_dir)

    import penrs_mcp_server
    importlib.reload(penrs_mcp_server)

    assert penrs_mcp_server.mcp.name == "penrs_mcp"

NameError: name 'pytest' is not defined

## Step 2 Tests

In [8]:
class DummyResponse:
    def __init__(
        self,
        status_code: int,
        *,
        json_data=None,
        text: str = "",
        json_error: Exception | None = None,
    ):
        self.status_code = status_code
        self._json_data = json_data
        self.text = text
        self._json_error = json_error

    @property
    def is_error(self) -> bool:
        return self.status_code >= 400

    def json(self):
        if self._json_error is not None:
            raise self._json_error
        return self._json_data


def _patch_async_client(monkeypatch, planned_results):
    instances = []

    class FakeAsyncClient:
        def __init__(self, timeout):
            self.timeout = timeout
            self.calls = []
            instances.append(self)

        async def __aenter__(self):
            return self

        async def __aexit__(self, exc_type, exc, tb):
            return False

        async def get(self, url, params=None, headers=None):
            self.calls.append({"url": url, "params": params, "headers": headers})
            if not planned_results:
                raise AssertionError("No planned response left for FakeAsyncClient.get()")

            next_result = planned_results.pop(0)
            if isinstance(next_result, Exception):
                raise next_result
            return next_result

    monkeypatch.setattr(penrs_http.httpx, "AsyncClient", FakeAsyncClient)
    return instances


def test_api_request_success_json_uses_default_timeout(monkeypatch):
    planned = [DummyResponse(200, json_data={"ok": True})]
    clients = _patch_async_client(monkeypatch, planned)

    result = asyncio.run(
        penrs_http._api_request(
            "https://example.test/data",
            params={"ticker": "MRNA"},
            headers={"X-Test": "1"},
            api_name="alpha_vantage",
        )
    )

    assert result == {"ok": True}
    assert len(clients) == 1
    assert clients[0].timeout == 30.0
    assert clients[0].calls == [
        {
            "url": "https://example.test/data",
            "params": {"ticker": "MRNA"},
            "headers": {"X-Test": "1"},
        }
    ]


def test_api_request_returns_text_when_json_parse_fails(monkeypatch):
    planned = [DummyResponse(200, text="raw text body", json_error=ValueError("bad json"))]
    _patch_async_client(monkeypatch, planned)

    result = asyncio.run(penrs_http._api_request("https://example.test/text"))

    assert result == {"text": "raw text body"}


def test_api_request_returns_structured_http_error_and_logs(monkeypatch):
    planned = [DummyResponse(500, text="server exploded")]
    _patch_async_client(monkeypatch, planned)

    with patch.object(penrs_http.logger, "error") as log_error:
        result = asyncio.run(penrs_http._api_request("https://example.test/fail", api_name="sec"))

    assert result == {"error": "HTTP 500", "detail": "server exploded"}
    assert log_error.call_count >= 1


def test_api_request_retries_429_503_with_exponential_backoff(monkeypatch):
    planned = [
        DummyResponse(429, text="rate limited"),
        DummyResponse(503, text="service unavailable"),
        DummyResponse(200, json_data={"status": "ok"}),
    ]
    clients = _patch_async_client(monkeypatch, planned)
    sleeps = []

    async def fake_sleep(seconds):
        sleeps.append(seconds)

    monkeypatch.setattr(penrs_http.asyncio, "sleep", fake_sleep)

    with patch.object(penrs_http.logger, "warning") as log_warning:
        result = asyncio.run(penrs_http._api_request("https://example.test/retry", api_name="ctgov"))

    assert result == {"status": "ok"}
    assert sleeps == [1, 2]
    assert len(clients[0].calls) == 3
    assert log_warning.call_count == 2


def test_api_request_stops_after_max_retries_on_429(monkeypatch):
    planned = [
        DummyResponse(429, text="limited"),
        DummyResponse(429, text="limited"),
        DummyResponse(429, text="limited"),
        DummyResponse(429, text="still limited"),
    ]
    clients = _patch_async_client(monkeypatch, planned)
    sleeps = []

    async def fake_sleep(seconds):
        sleeps.append(seconds)

    monkeypatch.setattr(penrs_http.asyncio, "sleep", fake_sleep)

    result = asyncio.run(penrs_http._api_request("https://example.test/limit", api_name="alpha"))

    assert result == {"error": "HTTP 429", "detail": "still limited"}
    assert len(clients[0].calls) == 4
    assert sleeps == [1, 2, 4]


def test_api_request_timeout_is_user_friendly_and_logged(monkeypatch):
    planned = [httpx.TimeoutException("took too long")]
    _patch_async_client(monkeypatch, planned)

    with patch.object(penrs_http.logger, "error") as log_error:
        result = asyncio.run(penrs_http._api_request("https://example.test/timeout", api_name="openfda"))

    assert result["error"] == "Request timed out"
    assert "https://example.test/timeout" in result["detail"]
    assert log_error.call_count >= 1


def test_api_request_request_error_is_user_friendly_and_logged(monkeypatch):
    request = httpx.Request("GET", "https://example.test/network")
    planned = [httpx.RequestError("network unreachable", request=request)]
    _patch_async_client(monkeypatch, planned)

    with patch.object(penrs_http.logger, "error") as log_error:
        result = asyncio.run(penrs_http._api_request("https://example.test/network", api_name="pubmed"))

    assert result["error"] == "Request failed"
    assert "network unreachable" in result["detail"]
    assert log_error.call_count >= 1


def test_api_request_respects_custom_timeout(monkeypatch):
    planned = [DummyResponse(200, json_data={"ok": True})]
    clients = _patch_async_client(monkeypatch, planned)

    result = asyncio.run(
        penrs_http._api_request(
            "https://example.test/custom-timeout",
            timeout=5.5,
            api_name="alpha_vantage",
        )
    )

    assert result == {"ok": True}
    assert clients[0].timeout == 5.5

## Worker Tests

In [9]:
def test_truncate_for_context_preserves_start_and_end():
    source = "A" * 80 + "MIDDLE" + "Z" * 80
    truncated = truncate_for_context(source, max_chars=60)

    assert len(truncated) == 60
    assert truncated.startswith("A")
    assert truncated.endswith("Z")
    assert "[truncated" in truncated


def test_parse_json_response_handles_markdown_block():
    worker = PENRSWorker(
        name="TestWorker",
        weight=1.0,
        signal_density=0.5,
        rubric_id="worker_test",
        document_type=DocumentType.SEC_10Q,
    )
    response = "analysis:\n```json\n{\"score\": 0.7, \"summary\": \"ok\"}\n```\nextra"

    parsed = worker.parse_json_response(response)

    assert parsed == {"score": 0.7, "summary": "ok"}


def test_parse_json_response_handles_embedded_prose_and_invalid_json_fallback():
    worker = PENRSWorker(
        name="TestWorker",
        weight=1.0,
        signal_density=0.5,
        rubric_id="worker_test",
        document_type=DocumentType.SEC_10Q,
    )
    prose_response = 'Before note. {"signal": "bearish", "confidence": 0.81} After note.'
    invalid_response = "I could not determine anything with confidence."

    parsed_from_prose = worker.parse_json_response(prose_response)
    parsed_invalid = worker.parse_json_response(invalid_response)

    assert parsed_from_prose == {"signal": "bearish", "confidence": 0.81}
    assert parsed_invalid["parse_error"] == "unable_to_parse_json"
    assert parsed_invalid["raw_response"] == invalid_response


def test_run_fetches_rubric_and_document_and_enriches_metadata():
    calls = {"rubric": 0, "document": 0, "prompt": None}

    async def fake_rubric_fetcher(rubric_id):
        calls["rubric"] += 1
        assert rubric_id == "worker_earnings"
        return {"name": "Earnings rubric", "threshold": 0.6}

    async def fake_document_fetcher(ticker, document_type, date_range):
        calls["document"] += 1
        assert ticker == "MRNA"
        assert document_type is DocumentType.EARNINGS_CALL
        assert date_range == {"from": "2026-01-01", "to": "2026-02-01"}
        return {
            "status": "available",
            "data": {
                "ticker": ticker,
                "sources": [{"api": "alpha_vantage", "data": {"transcript": "A" * 300}}],
            },
        }

    async def fake_llm(prompt):
        calls["prompt"] = prompt
        return "```json\n{\"score\": 0.42, \"thesis\": \"mixed\"}\n```"

    worker = PENRSWorker(
        name="EarningsWorker",
        weight=1.2,
        signal_density=0.75,
        rubric_id="worker_earnings",
        document_type=DocumentType.EARNINGS_CALL,
        rubric_fetcher=fake_rubric_fetcher,
        document_fetcher=fake_document_fetcher,
        llm_invoker=fake_llm,
        max_context_chars=120,
    )

    result = asyncio.run(worker.run("MRNA", "2026-01-01", "2026-02-01"))

    assert calls["rubric"] == 1
    assert calls["document"] == 1
    assert "Document excerpt:" in calls["prompt"]
    assert result["status"] == "available"
    assert result["worker"] == {
        "name": "EarningsWorker",
        "weight": 1.2,
        "signal_density": 0.75,
    }
    assert result["result"] == {"score": 0.42, "thesis": "mixed"}


def test_run_handles_not_released_without_llm_call():
    calls = {"llm": 0}

    async def fake_document_fetcher(_ticker, _document_type, _date_range):
        return {
            "status": "not_released",
            "data": {
                "apis_attempted": ["sec_edgar"],
                "reason": "Filing not yet published",
            },
        }

    async def fake_llm(_prompt):
        calls["llm"] += 1
        return {"score": 0.0}

    worker = PENRSWorker(
        name="SECFilingWorker",
        weight=1.0,
        signal_density=0.4,
        rubric_id="worker_sec",
        document_type=DocumentType.SEC_10Q,
        rubric_fetcher=lambda _rubric_id: {"stub": True},
        document_fetcher=fake_document_fetcher,
        llm_invoker=fake_llm,
    )

    result = asyncio.run(worker.run("BIIB", "2026-01-01", "2026-02-01"))

    assert result["status"] == "not_released"
    assert result["apis_attempted"] == ["sec_edgar"]
    assert result["worker"]["name"] == "SECFilingWorker"
    assert calls["llm"] == 0

## Test Runner

In [ ]:
# test_*.py source files have been removed; test functions are now defined inline above.
# Tests that require pytest fixtures (monkeypatch) need ipytest or a live pytest session.
# The tests below do not use fixtures and can be called directly:

import traceback

_direct_tests = [
    test_system_prompt_contains_required_role_and_mandatory_contradictions,
    test_arbiter_validates_required_worker_schema_fields,
    test_arbiter_clamps_scores_and_applies_star_rating_weights,
    test_arbiter_returns_json_schema_with_contradiction_flags_and_severities,
    test_document_type_is_strict_enum_with_nine_values,
    test_penrs_fetch_document_requires_document_type_enum,
    test_penrs_fetch_document_aggregates_multi_source_results,
    test_penrs_fetch_document_returns_not_released_with_attempted_apis,
    test_penrs_fetch_document_handles_partial_failures,
    test_truncate_for_context_preserves_start_and_end,
    test_parse_json_response_handles_markdown_block,
    test_parse_json_response_handles_embedded_prose_and_invalid_json_fallback,
    test_run_fetches_rubric_and_document_and_enriches_metadata,
    test_run_handles_not_released_without_llm_call,
    test_run_penrs_executes_pipeline_and_saves_report,
    test_run_penrs_worker_failure_is_isolated,
]

_passed = _failed = 0
for _fn in _direct_tests:
    try:
        _fn()
        print(f"PASS  {_fn.__name__}")
        _passed += 1
    except Exception as _exc:
        print(f"FAIL  {_fn.__name__}: {_exc}")
        traceback.print_exc()
        _failed += 1

print(f"\n{_passed} passed, {_failed} failed")
print("(Tests using monkeypatch require ipytest: pip install ipytest)")